In [36]:
import pandas as pd
import numpy as np

# 1. 读取所有子表
file_path = '1.xls'
xls = pd.ExcelFile(file_path)

sheet_names = xls.sheet_names[:13]
dfs = []

for sheet in sheet_names:
    df = xls.parse(sheet)
    
    # 2. 去除列名的空格、全角空格
    df.columns = df.columns.str.strip().str.replace(' ', '').str.replace(' ', '')
    
    # 3. 统一缺失值（如 NA、空字符串）处理
    df.replace(['NA', 'NaN', '', ' '], np.nan, inplace=True)

    # 4. 转换数值字段（非数值会变成 NaN）
    num_cols = ['SO2', 'NO2', 'PM10', 'CO', 'O31小时', 'O38小时', 'PM2.5']
    for col in num_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')  # 强制转换为数值

    # 5. 转换时间字段
    if '时间' in df.columns:
        df['时间'] = pd.to_datetime(df['时间'], errors='coerce')

    # 6. 加上 sheet 名为“区域”字段
    df['来源Sheet'] = sheet

    dfs.append(df)

# 7. 合并所有子表
data = pd.concat(dfs, ignore_index=True)

# 8. 去除空行（全为空的）
data.dropna(how='all', inplace=True)

# 9. （可选）处理异常值，例如负数、超过阈值的污染物
for col in num_cols:
    if col in data.columns:
        data = data[(data[col].isna()) | (data[col] >= 0)]  # 去除负数
data
# 10. 删掉列"来源Sheet"和"序号"
data.drop(columns=['来源Sheet', '序号'], inplace=True, errors='ignore')
# 11. 重新保存为 Excel 文件

data.to_csv('仅换NA为空.csv', index=False)


In [38]:
# 将时间列为空的替换为前后非空值的平均值
if '时间' in data.columns:
    data['时间'] = data['时间'].fillna(method='ffill').fillna(method='bfill')
# 12. 保存最终结果
data.to_csv('时间矫正.csv', index=False)

C:\Users\金田泉\AppData\Local\Temp\ipykernel_2500\1290004224.py:3: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data['时间'] = data['时间'].fillna(method='ffill').fillna(method='bfill')


PermissionError: [Errno 13] Permission denied: '时间矫正.csv'

In [ ]:
# 将23456789列全为空的行删除
data.dropna(subset=['SO2', 'NO2', 'PM10', 'CO', 'O31小时', 'O38小时', 'PM2.5',"首要污染物"], how='all', inplace=True)
# 13. 最终保存
data.to_csv('删掉全空.csv', index=False)

PermissionError: [Errno 13] Permission denied: '删掉全空.csv'

In [ ]:
# 将同时缺失'CO', 'O31小时', 'O38小时', 'PM2.5'列分为一个子表,其他情况分为另外一个子表
mask = data[['CO', 'O31小时', 'O38小时', 'PM2.5']].isna().all(axis=1)
data_with_all_na = data[mask]
data_with_other = data[~mask]
# 14. 保存两个子表
data_with_all_na.to_csv('CO_O31_O38_PM2.5全空.csv', index=False)
data_with_other.to_csv('全有.csv', index=False)

PermissionError: [Errno 13] Permission denied: 'CO_O31_O38_PM2.5全空.csv'

In [42]:
import numpy as np
import pandas as pd

def iterative_svd_impute(dataframe, k=3, max_iter=50, tol=1e-4, verbose=True):
    """
    使用迭代式 SVD 对缺失值进行填补
    :param dataframe: 输入 DataFrame，数值型
    :param k: 保留的奇异值个数
    :param max_iter: 最大迭代次数
    :param tol: 收敛误差阈值
    :param verbose: 是否打印误差
    :return: 填补后的 DataFrame
    """
    df = dataframe.copy()
    mask_missing = df.isnull().values
    X = df.values.astype(float)

    # Step 1: 用列均值初始化缺失值
    col_means = np.nanmean(X, axis=0)
    X[np.isnan(X)] = np.take(col_means, np.where(np.isnan(X))[1])

    last_X = X.copy()

    for iteration in range(max_iter):
        # Step 2: SVD 分解
        U, s, Vt = np.linalg.svd(X, full_matrices=False)

        # Step 3: 截断重构
        S = np.diag(s[:k])
        X_reconstructed = np.dot(U[:, :k], np.dot(S, Vt[:k, :]))

        # Step 4: 仅替换缺失值位置
        X[mask_missing] = X_reconstructed[mask_missing]

        # Step 5: 判断收敛（与上次的变化程度）
        diff = np.linalg.norm(X - last_X) / np.linalg.norm(last_X)
        if verbose:
            print(f"第 {iteration + 1} 次迭代，变化率: {diff:.6f}")
        if diff < tol:
            break

        last_X = X.copy()

    # 构造返回 DataFrame
    df_filled = pd.DataFrame(X, columns=df.columns, index=df.index)
    return df_filled
# 15. 对全有子表进行 SVD 填补
cols = ['SO2', 'NO2', 'PM10', 'CO', 'O31小时', 'O38小时', 'PM2.5']
# 这里直接读取 '全有.csv' 文件并进行 SVD 填补
data_with_other = pd.read_csv('全有.csv')
data_with_other_filled = iterative_svd_impute(data_with_other[cols], k=3, max_iter=100)


第 1 次迭代，变化率: 0.018684
第 2 次迭代，变化率: 0.006840
第 3 次迭代，变化率: 0.004159
第 4 次迭代，变化率: 0.003409
第 5 次迭代，变化率: 0.003140
第 6 次迭代，变化率: 0.003006
第 7 次迭代，变化率: 0.002916
第 8 次迭代，变化率: 0.002843
第 9 次迭代，变化率: 0.002779
第 10 次迭代，变化率: 0.002719
第 11 次迭代，变化率: 0.002663
第 12 次迭代，变化率: 0.002611
第 13 次迭代，变化率: 0.002560
第 14 次迭代，变化率: 0.002513
第 15 次迭代，变化率: 0.002467
第 16 次迭代，变化率: 0.002423
第 17 次迭代，变化率: 0.002381
第 18 次迭代，变化率: 0.002341
第 19 次迭代，变化率: 0.002302
第 20 次迭代，变化率: 0.002264
第 21 次迭代，变化率: 0.002227
第 22 次迭代，变化率: 0.002191
第 23 次迭代，变化率: 0.002155
第 24 次迭代，变化率: 0.002121
第 25 次迭代，变化率: 0.002087
第 26 次迭代，变化率: 0.002054
第 27 次迭代，变化率: 0.002021
第 28 次迭代，变化率: 0.001989
第 29 次迭代，变化率: 0.001957
第 30 次迭代，变化率: 0.001926
第 31 次迭代，变化率: 0.001895
第 32 次迭代，变化率: 0.001865
第 33 次迭代，变化率: 0.001834
第 34 次迭代，变化率: 0.001805
第 35 次迭代，变化率: 0.001775
第 36 次迭代，变化率: 0.001746
第 37 次迭代，变化率: 0.001717
第 38 次迭代，变化率: 0.001689
第 39 次迭代，变化率: 0.001661
第 40 次迭代，变化率: 0.001633
第 41 次迭代，变化率: 0.001606
第 42 次迭代，变化率: 0.001579
第 43 次迭代，变化率: 0.001552
第 44 次迭代，变化率: 0.0015

In [ ]:
# 查看填补后的数据
print(data_with_other_filled.head())
# 将填补后的数据合并回原 DataFrame
data_with_other[cols] = data_with_other_filled
# 16. 保存填补后的数据
data_with_other.to_csv('全有_SVD填补.csv', index=False)


         SO2         NO2   PM10    CO  O31小时  O38小时  PM2.5
0  23.000000   79.000000  140.0  42.0  107.0  116.0  133.0
1  26.000000   88.000000  167.0  46.0   95.0   99.0  169.0
2  80.178993  130.335435  154.0  77.0   59.0   78.0  195.0
3  24.000000   82.000000  133.0  79.0   40.0   44.0  200.0
4  13.000000   45.000000  125.0  71.0    9.0    9.0  220.0


PermissionError: [Errno 13] Permission denied: '全有_SVD填补.csv'

In [50]:
# 分析CO_O31_O38_PM2.5全空.csv 的"首要污染物"列
def analyze_primary_pollutant(df):
    """
    分析首要污染物列的分布情况
    :param df: 输入 DataFrame
    :return: None
    """
    if '首要污染物' not in df.columns:
        print("DataFrame中没有'首要污染物'列")
        return

    # 统计各个首要污染物的出现次数
    pollutant_counts = df['首要污染物'].value_counts()

    # 打印统计结果
    print("首要污染物分布情况：")
    print(pollutant_counts)
# 读取 CO_O31_O38_PM2.5全空.csv 文件
co_o31_o38_pm25_empty = pd.read_csv('CO_O31_O38_PM2.5全空.csv')
# 分析首要污染物列
analyze_primary_pollutant(co_o31_o38_pm25_empty)
# 将首要污染物列的"---"填充为空
co_o31_o38_pm25_empty['首要污染物'] = co_o31_o38_pm25_empty['首要污染物'].replace('---', np.nan)
#将所有为0的项填充为空
co_o31_o38_pm25_empty[cols] = co_o31_o38_pm25_empty[cols].replace(0, np.nan)
# 将SO2	NO2	PM10	CO	O31小时	O38小时	PM2.5	首要污染物全空的行删除
co_o31_o38_pm25_empty.dropna(subset=['SO2', 'NO2', 'PM10', 'CO', 'O31小时', 'O38小时', 'PM2.5', '首要污染物'], how='all', inplace=True)
# 保存处理后的数据
co_o31_o38_pm25_empty.to_csv('CO_O31_O38_PM2.5全空_处理后.csv', index=False)
# 分析处理后的数据
analyze_primary_pollutant(co_o31_o38_pm25_empty)

首要污染物分布情况：
首要污染物
良        9854
轻微污染     1860
优         884
轻度污染      328
中度污染       44
轻微         44
中度重污染      41
---        23
重污染        19
重度污染        9
Name: count, dtype: int64
首要污染物分布情况：
首要污染物
良        9854
轻微污染     1860
优         884
轻度污染      328
中度污染       44
轻微         44
中度重污染      41
重污染        19
重度污染        9
Name: count, dtype: int64


In [53]:
#读入数据
co_o31_o38_pm25_empty = pd.read_csv('CO_O31_O38_PM2.5全空_处理后.csv')
# SVD 填补 CO_O31_O38_PM2.5全空_处理后.csv 的数据
co_o31_o38_pm25_empty_filled = iterative_svd_impute(co_o31_o38_pm25_empty[cols], k=3, max_iter=100)
# 将填补后的数据合并回原 DataFrame
co_o31_o38_pm25_empty[cols] = co_o31_o38_pm25_empty_filled
# 保存填补后的数据
co_o31_o38_pm25_empty.to_csv('CO_O31_O38_PM2.5全空_处理后_SVD填补.csv', index=False)

第 1 次迭代，变化率: 0.000000


In [54]:
#读入全有数据
data_with_other = pd.read_csv('全有_SVD填补.csv')
#将所有负数都置为0
data_with_other[cols] = data_with_other[cols].clip(lower=0)
#保存处理后的数据
data_with_other.to_csv('全有_SVD填补_负数置0.csv', index=False)